## Step 0 
Loading the libraries and defines small helper functions.

In [1]:
"Assessment 2 : Hotel Booking analysis pipeline. Usage:hotel_bookings.csv from. Produce: Number Table(CSV formate), PNG , log.txt. Requires: pandas numpy scipy scikit-learn statsmodels matplotlib "
import sys,os,warnings
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
from sklearn.linear_model import LogisticRegression,LinearRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier,RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score,precision_score,recall_score,f1_score,roc_auc_score,average_precision_score,confusion_matrix,roc_curve,precision_recall_curve,r2_score,mean_squared_error,mean_absolute_error)
warnings.filterwarnings("ignore")

SRC="hotel_bookings.csv" #original dataset taken from kaggel
OUT = "outputs"
os.makedirs(OUT, exist_ok=True)
SEED=1
LOG=open(os.path.join(OUT,"analysis_log.txt"),"w")

def log(*a):
    s="".join(str(x) for x in a)  
    print(s); LOG.write(s + "\n")
    
def table(df,name, index=True):
    df.to_csv(os.path.join(OUT, name), index=index)
    
def fig(name):
    plt.tight_layout(); plt.savefig(os.path.join(OUT,name),dpi=200);plt.sho()
    
MONTHS = ["January","February","March","April","May","June","July","August",
          "September","October","November","December"]
MNUM={m:i+1 for i,m in enumerate(MONTHS)}



## Step 1- Clean the data
**Objective:** Removing exact duplicates, zero-guest bookings,zero-night stays, negative or extreme ADR (average daily rate), and undefined values.
**Log**: Log are visible at log file for removed rows
**Quoted in report**: The counts are reported in assessment.

In [4]:
#-----------------------------------------------------------1. CLEAN
raw=pd.read_csv(SRC)
log("== 1. DATA CLEANING==")
log("Raw rows/cols:", raw.shape)
log("Raw cancellation rate: %.3f" % raw.is_canceled.mean())
df = raw.drop_duplicates()
steps=[("Raw rows", len(raw))]
df= df.copy(); df["children"]=df["children"].fillna(0)
guests=df.adults + df.children + df.babies
nights=df.stays_in_weekend_nights + df.stays_in_week_nights
df=df[guests > 0]; steps.append(("After removing zero-guest bookings",len(df)))
nights=df.stays_in_weekend_nights + df.stays_in_week_nights
df =df[nights > 0];
steps.append(("After removing zero-night stays", len(df)))
df=df[(df.adr >=0) & (df.adr <=1000)]; steps.append(("After removing exact duplicates", len(df)))
df = df[df.meal != "Undefined"];    steps.append(("After removing undefined meal", len(df)))
df=df[(df.market_segment !="Undefined") & (df.distribution_channel !="Undefined")]
steps.append(("After removing undefined segment / channel",len(df)))
clean_log=pd.DataFrame(steps, columns=["Step","Rows"])
clean_log["Removed"]=(-clean_log.Rows.diff()).fillna(0).astype(int)
table(clean_log, "table_01_cleaning_log.csv", index=False)
log(clean_log.to_string(index=False))
log("Cleaned cancellation rate: %.3f" % df.is_canceled.mean())
log("ADR==0 rows kept for Problem 1, excluded from Problem 2:",(df.adr==0).sum())

COLS = ["is_canceled","adr","hotel","lead_time","arrival_date_year","arrival_date_month","stays_in_weekend_nights","stays_in_week_nights","adults","children","babies","meal", "market_segment", "distribution_channel", "is_repeated_guest","previous_cancellations", "deposit_type", "customer_type", "reserved_room_type","total_of_special_requests"]
d = df[COLS].copy()
d["month_num"] = d.arrival_date_month.map(MNUM)
d["total_nights"] = d.stays_in_weekend_nights + d.stays_in_week_nights
d.to_csv(os.path.join(OUT, "hotel_clean_20cols.csv"), index=False)
log("Clean analysis file:", d.shape[0], "rows,", len(COLS), "columns")


== 1. DATA CLEANING==
Raw rows/cols:(119390, 32)
Raw cancellation rate: 0.370
                                      Step   Rows  Removed
                                  Raw rows 119390        0
        After removing zero-guest bookings  87230    32160
           After removing zero-night stays  86639      591
           After removing exact duplicates  86637        2
             After removing undefined meal  86152      485
After removing undefined segment / channel  86147        5
Cleaned cancellation rate: 0.277
ADR==0 rows kept for Problem 1, excluded from Problem 2:1029
Clean analysis file:86147rows,20columns


## Step 2 - Descriptive analytics

**What it does:** Summary statistics for the numeric variables, cancellation rate by hotel,segment,deposit type and lead time, and median ADR by month and hotel.

**Observed:** Observing Figure 1-3 and the cancellation-rate tables. Groups cancel the most and ADR changes with season.

**Report:** The figures are explained in the report with respect to what it shows and why it matters to managers.

In [13]:
#---------DESCRIPTIVE
log("\n== 2. DESCRIPTIVE ANALYTICS ==")
num=["lead_time,""stay_in_weekend_nights","stays_in_week_nights","adults","children","babes","previous_cancellations","total_of_special_requests","adr"]
num = [
    "lead_time",
    "stays_in_weekend_nights",
    "stays_in_week_nights",
    "adults",
    "children",
    "babies",
    "previous_cancellations",
    "total_of_special_requests",
    "adr",
]

desc = d[num].describe().T[
    ["mean", "std", "min", "25%", "50%", "75%", "max"]
].round(2)
table(desc,"table_02_numeric_summary.csv");
log(desc.to_string())

def rate(col):
    t=d.groupby(col).is_canceled.agg(Bookings="size",Cancel_rate="mean").round(3)
    return t

cancel_tables = {c: rate(c) for c in ["hotel", "market_segment", "deposit_type", "customer_type",
                                      "distribution_channel", "meal", "arrival_date_year"]}
for c, t in cancel_tables.items():
        table(t, f"table_03_cancel_rate_by_{c}.csv"); log("\nCancellation rate by", c); log(t.to_string())
        
        #Fig 1 cancellation by segment & deposit
        fig_, ax=plt.subplots(1,2,figsize=(11,4))
for a,c in zip(ax,["market_segment","deposit_type"]):
            t=cancel_tables[c]["Cancel_rate"].sort_values()
            a.barh(t.index, t.values * 100, color="#2b6cb0")
            a.set_xlabel("Cancellatioin rate (%)");
            a.set_title(f"Cancellation rate by {c.replace('_','')}")
def fig(name):
    plt.tight_layout()
    plt.savefig(os.path.join(OUT, name), dpi=200)
    plt.close()
        
        # Fig 2 lead time vs cancellation
d["lead_band"]=pd.cut(d.lead_time, [-1,7,30,90,180,365,800],labels=["0-7", "8-30", "31-90", "91-180", "181-365", "366+"])
lb = d.groupby("lead_band", observed=True).is_canceled.mean() * 100
table(d.groupby("lead_band", observed=True).is_canceled.agg(Bookings="size", Cancel_rate="mean").round(3),"table_04_cancel_rate_by_lead_band.csv")
plt.figure(figsize=(6, 4)); plt.bar(lb.index.astype(str), lb.values, color="#2b6cb0")
plt.xlabel("Lead time (days)"); plt.ylabel("Cancellation rate (%)"); plt.title("Cancellation rate rises with lead time")
fig("figure_02_cancel_by_lead_time.png")               

 # Fig 3 ADR by month and hotel
adr = d[d.adr > 0].groupby(["month_num", "hotel"]).adr.median().unstack()
adr.index = [MONTHS[i - 1][:3] for i in adr.index]
table(adr.round(1), "table_05_median_adr_by_month_hotel.csv")
adr.plot(marker="o", figsize=(7, 4), color=["#2b6cb0", "#dd6b20"])
plt.ylabel("Median ADR"); plt.xlabel("Arrival month"); plt.title("Median ADR by month and hotel (ADR > 0)")
fig("figure_03_adr_by_month_hotel.png")
log("\nMedian ADR by month and hotel:\n", adr.round(0).to_string())           
                


== 2. DESCRIPTIVE ANALYTICS ==
                             mean    std  min   25%   50%     75%    max
lead_time                   80.24  86.10  0.0  12.0  50.0  125.00  709.0
stays_in_weekend_nights      1.01   1.03  0.0   0.0   1.0    2.00   19.0
stays_in_week_nights         2.63   2.03  0.0   1.0   2.0    4.00   50.0
adults                       1.88   0.62  0.0   2.0   2.0    2.00   55.0
children                     0.14   0.46  0.0   0.0   0.0    0.00   10.0
babies                       0.01   0.11  0.0   0.0   0.0    0.00   10.0
previous_cancellations       0.03   0.37  0.0   0.0   0.0    0.00   26.0
total_of_special_requests    0.70   0.83  0.0   0.0   1.0    1.00    5.0
adr                        107.19  51.18  0.0  73.0  99.0  134.35  510.0

Cancellation rate byhotel
              Bookings  Cancel_rate
hotel                              
City Hotel       53038        0.302
Resort Hotel     33109        0.238

Cancellation rate bymarket_segment
                Bookings  Cance

## Step 2b Descriptive Analysis for ADR Room Price

In [18]:
# ------------------------------------------------------------- 2b. DESCRIPTIVE - ADR (Problem 2)
log("\n== 2b. DESCRIPTIVE ANALYTICS - ADR (average daily rate) ==")
zero_adr = int((d.adr == 0).sum())
a = d[d.adr > 0].copy()
log("Bookings with ADR = 0 (excluded here, often complimentary stays):", zero_adr)
log("Bookings used for ADR analysis:", len(a))

# 1. Overall distribution
adr_desc = a.adr.describe(percentiles=[.05, .25, .5, .75, .95]).round(2).to_frame("ADR")
adr_desc.loc["skewness"] = round(float(stats.skew(a.adr)), 2)
table(adr_desc, "table_16_adr_distribution.csv"); log(adr_desc.to_string())
by_hotel = a.groupby("hotel").adr.agg(Bookings="size", Mean="mean", Median="median", Std="std",
                                      P25=lambda s: s.quantile(.25), P75=lambda s: s.quantile(.75)).round(1)
table(by_hotel, "table_17_adr_by_hotel.csv"); log("\nADR by hotel\n", by_hotel.to_string())

fig_, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].hist(a.adr, bins=60, range=(0, 400), color="#2b6cb0")
ax[0].axvline(a.adr.mean(), color="red", ls="--", label=f"Mean {a.adr.mean():.0f}")
ax[0].axvline(a.adr.median(), color="black", ls="--", label=f"Median {a.adr.median():.0f}")
ax[0].set_xlabel("ADR"); ax[0].set_ylabel("Bookings"); ax[0].set_title("Distribution of ADR (right-skewed)"); ax[0].legend()
for h, c in zip(["City Hotel", "Resort Hotel"], ["#2b6cb0", "#dd6b20"]):
    ax[1].hist(a[a.hotel == h].adr, bins=60, range=(0, 400), alpha=.55, color=c, label=h)
ax[1].set_xlabel("ADR"); ax[1].set_title("ADR by hotel"); ax[1].legend()
fig("figure_09_adr_distribution.png")

# 2. ADR by category (mean, median, spread)
cat_cols = ["market_segment", "customer_type", "reserved_room_type", "deposit_type", "meal",
            "distribution_channel", "is_repeated_guest", "arrival_date_year"]
for c in cat_cols:
    t = a.groupby(c).adr.agg(Bookings="size", Mean="mean", Median="median", Std="std").round(1)
    t = t.sort_values("Median", ascending=False)
    table(t, f"table_18_adr_by_{c}.csv"); log(f"\nADR by {c}\n", t.to_string())

def box(ax, col, title, min_n=200):
    g = a.groupby(col).adr
    keep = g.size()[g.size() >= min_n].index
    order = g.median()[keep].sort_values().index.tolist()
    ax.boxplot([a[a[col] == k].adr for k in order], showfliers=False)
    ax.set_xticks(range(1, len(order) + 1))
    ax.set_xticklabels([str(k) for k in order], rotation=45, ha="right")
    ax.set_ylabel("ADR"); ax.set_title(title)

fig_, ax = plt.subplots(1, 2, figsize=(12, 4.5))
box(ax[0], "market_segment", "ADR by market segment (groups with 200+ bookings)")
box(ax[1], "reserved_room_type", "ADR by reserved room type")
fig("figure_10_adr_by_segment_and_room.png")

# 3. ADR against stay and guest characteristics
a["lead_band"] = pd.cut(a.lead_time, [-1, 7, 30, 90, 180, 365, 800], labels=["0-7", "8-30", "31-90", "91-180", "181-365", "366+"])
a["nights_band"] = pd.cut(a.total_nights, [0, 1, 3, 7, 14, 100], labels=["1", "2-3", "4-7", "8-14", "15+"])
a["party"] = (a.adults + a.children + a.babies).clip(upper=5).astype(int).astype(str).replace({"5": "5+"})
fig_, ax = plt.subplots(1, 3, figsize=(14, 4))
for axx, col, ttl in zip(ax, ["lead_band", "nights_band", "party"],
                         ["Lead time (days)", "Length of stay (nights)", "Guests in booking"]):
    t = a.groupby(col, observed=True).adr.agg(Bookings="size", Median="median", Mean="mean").round(1)
    table(t, f"table_19_adr_by_{col}.csv")
    axx.bar(t.index.astype(str), t.Median, color="#2b6cb0"); axx.set_xlabel(ttl); axx.set_ylabel("Median ADR")
    axx.set_title(f"Median ADR by {ttl.lower()}")
    log(f"\nMedian ADR by {col}\n", t.to_string())
fig("figure_11_adr_by_stay_characteristics.png")

# 4. Monthly pattern: mean ADR heatmap (hotel x month) and correlations
hm = a.pivot_table(index="hotel", columns="month_num", values="adr", aggfunc="mean").round(0)
hm.columns = [MONTHS[i - 1][:3] for i in hm.columns]
table(hm, "table_20_mean_adr_hotel_by_month.csv"); log("\nMean ADR, hotel by month\n", hm.to_string())
plt.figure(figsize=(9, 2.4)); plt.imshow(hm.values, cmap="Blues", aspect="auto")
plt.xticks(range(12), hm.columns); plt.yticks(range(len(hm.index)), hm.index)
for i in range(hm.shape[0]):
    for j in range(hm.shape[1]):
        plt.text(j, i, int(hm.values[i, j]), ha="center", va="center", fontsize=8)
plt.title("Mean ADR by hotel and arrival month"); fig("figure_12_adr_heatmap_month_hotel.png")

rows = []
for c in ["lead_time", "total_nights", "adults", "children", "babies", "previous_cancellations", "total_of_special_requests"]:
    rho, p = stats.spearmanr(a[c], a.adr); rows.append([c, round(rho, 3), p])
corr = pd.DataFrame(rows, columns=["Variable", "Spearman_rho_with_ADR", "p_value"]).sort_values("Spearman_rho_with_ADR", key=abs, ascending=False)
table(corr, "table_21_adr_spearman_correlations.csv", index=False); log("\nSpearman correlation with ADR\n", corr.to_string(index=False))
log("\nADR descriptive done.")


== 2b. DESCRIPTIVE ANALYTICS - ADR (average daily rate) ==
Bookings with ADR = 0 (excluded here, often complimentary stays):1029
Bookings used for ADR analysis:85118
               ADR
count     85118.00
mean        108.48
std          50.10
min           0.26
5%           41.47
25%          74.75
50%          99.00
75%         135.00
95%         205.00
max         510.00
skewness      1.04

ADR by hotel
              Bookings   Mean  Median   Std   P25    P75
hotel                                                   
City Hotel       52316  113.2   106.2  39.5  85.0  135.0
Resort Hotel     32802  101.0    80.0  62.7  53.0  135.0

ADR by market_segment
                Bookings   Mean  Median   Std
market_segment                               
Online TA          51228  119.0   111.2  48.8
Direct             11498  118.3   108.0  57.9
Aviation             222  102.4    95.0  14.7
Offline TA/TO      13464   82.7    80.0  35.0
Groups              4513   78.1    70.0  35.2
Corporate         

## Step-3 Diagnostic Analysis


In [14]:
# ------------------------------------------------------------- 3. DIAGNOSTIC
log("\n== 3. DIAGNOSTIC ANALYTICS ==")
rows = []
for c in ["hotel", "arrival_date_month", "meal", "market_segment", "distribution_channel",
          "is_repeated_guest", "deposit_type", "customer_type", "reserved_room_type"]:
    ct = pd.crosstab(d[c], d.is_canceled)
    chi2, p, dof, _ = stats.chi2_contingency(ct)
    v = np.sqrt(chi2 / (ct.values.sum() * (min(ct.shape) - 1)))
    rows.append([c, round(chi2, 1), dof, p, round(v, 3)])
chi = pd.DataFrame(rows, columns=["Variable", "Chi-square", "df", "p-value", "Cramers_V"]).sort_values("Cramers_V", ascending=False)
table(chi, "table_06_chi_square_tests.csv", index=False); log(chi.to_string(index=False))

rows = []
for c in ["lead_time", "total_nights", "adults", "previous_cancellations", "total_of_special_requests", "adr"]:
    a, b = d[d.is_canceled == 1][c], d[d.is_canceled == 0][c]
    u, p = stats.mannwhitneyu(a, b)
    rows.append([c, round(a.median(), 1), round(b.median(), 1), p])
mw = pd.DataFrame(rows, columns=["Variable", "Median_cancelled", "Median_kept", "MannWhitney_p"])
table(mw, "table_07_numeric_vs_cancel.csv", index=False); log(mw.to_string(index=False))

# Logistic regression odds ratios (statsmodels) on full clean data
lr_cols = ["lead_time", "total_nights", "adults", "children", "babies", "is_repeated_guest",
           "previous_cancellations", "total_of_special_requests", "hotel", "deposit_type",
           "market_segment", "customer_type"]
Xo = pd.get_dummies(d[lr_cols], drop_first=True).astype(float)
Xo = sm.add_constant(Xo)
logit = sm.Logit(d.is_canceled, Xo).fit(disp=0)
orr = pd.DataFrame({"Odds_ratio": np.exp(logit.params), "CI_low": np.exp(logit.conf_int()[0]),
                    "CI_high": np.exp(logit.conf_int()[1]), "p_value": logit.pvalues}).drop("const").round(3)
orr = orr.sort_values("Odds_ratio", ascending=False)
table(orr, "table_08_logit_odds_ratios.csv"); log(orr.to_string())
log("Logit pseudo R2: %.3f" % logit.prsquared)
plot = orr[~orr.index.str.startswith("deposit_type_Non")].copy()
plt.figure(figsize=(7, 6)); y = range(len(plot))
plt.errorbar(plot.Odds_ratio, y, xerr=[plot.Odds_ratio - plot.CI_low, plot.CI_high - plot.Odds_ratio], fmt="o", color="#2b6cb0")
plt.axvline(1, color="grey", ls="--"); plt.xscale("log"); plt.yticks(list(y), plot.index, fontsize=7)
plt.xlabel("Odds ratio (log scale)"); plt.title("Drivers of cancellation (non-refundable deposit omitted for scale)")
fig("figure_04_odds_ratios.png")



== 3. DIAGNOSTIC ANALYTICS ==
            Variable  Chi-square  df       p-value  Cramers_V
      market_segment      4215.3   6  0.000000e+00      0.221
        deposit_type      2329.1   2  0.000000e+00      0.164
distribution_channel      2014.0   3  0.000000e+00      0.153
       customer_type      1401.5   3 1.375277e-303      0.128
   is_repeated_guest       617.5   1 2.679109e-136      0.085
  arrival_date_month       618.9  11 1.293700e-125      0.085
               hotel       415.3   1  2.614226e-92      0.069
                meal       328.8   3  5.826141e-71      0.062
  reserved_room_type       275.4   8  6.858550e-55      0.057
                 Variable  Median_cancelled  Median_kept  MannWhitney_p
                lead_time              80.0         38.0   0.000000e+00
             total_nights               3.0          3.0  1.730119e-192
                   adults               2.0          2.0  6.058577e-150
   previous_cancellations               0.0          0.0  3.7

## Step 4a Predictive Analysis


In [15]:
#--------------------------------------------------------------------4q. PREDICTIVE P1
log("\n===4a. PREDICTIVE CANCELLATION (time-based split: train 2015-16, test 2017)==")
feat1 = ["hotel", "lead_time", "month_num", "stays_in_weekend_nights", "stays_in_week_nights", "adults",
         "children", "babies", "meal", "market_segment", "distribution_channel", "is_repeated_guest",
         "previous_cancellations", "deposit_type", "customer_type", "reserved_room_type",
         "total_of_special_requests"]
tr=d.arrival_date_year < 2017;
te=~tr
y=d.is_canceled
log("Train rows %d, test rows %d, test cancel rate %.3f" % (tr.sum(), te.sum(),y[te].mean()))

def metrics(name, yt, p, thr=0.5):
    pr=(p>=thr).astype(int)
    return dict(Model=name,Accuracy=accuracy_score(yt,pr),Precision=precision_score(yt,pr),Recall=recall_score(yt,pr),F1=f1_score(yt,pr),ROC_AUC=roc_auc_score(yt,p),PR_AUC=average_precision_score(yt,p))

def fit_models(features, tag):
    X = pd.get_dummies(d[features], drop_first=True).astype(float)
    sc = StandardScaler().fit(X[tr])
    models = {"Logistic regression": (LogisticRegression(max_iter=3000), True),
              "Decision tree (depth 6)": (DecisionTreeClassifier(max_depth=6, random_state=SEED), False),
              "Random forest": (RandomForestClassifier(300, min_samples_leaf=5, n_jobs=-1, random_state=SEED), False)}
    res, probs, fitted = [], {}, {}
    base = dict(Model="Majority-class baseline", Accuracy=1 - y[te].mean(), Precision=np.nan, Recall=0.0,
                F1=np.nan, ROC_AUC=0.5, PR_AUC=y[te].mean())
    res.append(base)
    for n, (m, scaled) in models.items():
        Xt, Xe = (sc.transform(X[tr]), sc.transform(X[te])) if scaled else (X[tr], X[te])
        m.fit(Xt, y[tr]); p = m.predict_proba(Xe)[:, 1]
        res.append(metrics(n, y[te], p)); probs[n] = p; fitted[n] = m
    r = pd.DataFrame(res).round(3)
    table(r, f"table_09_cancel_model_metrics_{tag}.csv", index=False)
    log(f"\nCancellation models ({tag}), threshold 0.5:\n", r.to_string(index=False))
    return X, probs, fitted, r

X1, probs1, fit1, res_with = fit_models(feat1, "with_deposit")
_, _, _, res_without = fit_models([f for f in feat1 if f != "deposit_type"], "without_deposit")

best = "Random forest"; p = probs1[best]; yt = y[te].values
cm = confusion_matrix(yt, (p >= .5).astype(int))
table(pd.DataFrame(cm, index=["Actual kept", "Actual cancelled"], columns=["Pred kept", "Pred cancelled"]),
      "table_10_confusion_matrix_rf.csv"); log("\nConfusion matrix (RF, 0.5):\n", cm)

fig_, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].imshow(cm, cmap="Blues")
for i in range(2):
    for j in range(2): ax[0].text(j, i, f"{cm[i,j]:,}", ha="center", va="center")
ax[0].set_xticks([0, 1], ["Pred kept", "Pred cancelled"]); ax[0].set_yticks([0, 1], ["Kept", "Cancelled"])
ax[0].set_title("Confusion matrix - random forest, test 2017")
for n, pp in probs1.items():
    fpr, tpr, _ = roc_curve(yt, pp); ax[1].plot(fpr, tpr, label=f"{n} ({roc_auc_score(yt,pp):.2f})")
    pr_, rc_, _ = precision_recall_curve(yt, pp); ax[2].plot(rc_, pr_, label=n)
ax[1].plot([0, 1], [0, 1], "k--", lw=.8); ax[1].set_xlabel("False positive rate"); ax[1].set_ylabel("True positive rate")
ax[1].set_title("ROC curves"); ax[1].legend(fontsize=7)
ax[2].axhline(yt.mean(), color="k", ls="--", lw=.8); ax[2].set_xlabel("Recall"); ax[2].set_ylabel("Precision")
ax[2].set_title("Precision-recall curves (dashed = baseline)")
fig("figure_05_cancel_model_evaluation.png")

imp = pd.Series(fit1[best].feature_importances_, index=X1.columns).sort_values(ascending=False).head(12)
table(imp.round(4).to_frame("Importance"), "table_11_rf_feature_importance.csv")
plt.figure(figsize=(6, 4)); imp[::-1].plot.barh(color="#2b6cb0"); plt.title("Random forest: top 12 features")
fig("figure_06_feature_importance.png")


===4a. PREDICTIVE CANCELLATION (time-based split: train 2015-16, test 2017)==
Train rows 54837, test rows 31310, test cancel rate 0.321

Cancellation models (with_deposit), threshold 0.5:
                  Model  Accuracy  Precision  Recall    F1  ROC_AUC  PR_AUC
Majority-class baseline     0.679        NaN   0.000   NaN    0.500   0.321
    Logistic regression     0.729      0.676   0.302 0.417    0.746   0.598
Decision tree (depth 6)     0.736      0.637   0.415 0.503    0.755   0.557
          Random forest     0.735      0.639   0.402 0.493    0.763   0.595

Cancellation models (without_deposit), threshold 0.5:
                  Model  Accuracy  Precision  Recall    F1  ROC_AUC  PR_AUC
Majority-class baseline     0.679        NaN   0.000   NaN    0.500   0.321
    Logistic regression     0.724      0.652   0.301 0.412    0.739   0.569
Decision tree (depth 6)     0.728      0.623   0.389 0.479    0.743   0.533
          Random forest     0.732      0.636   0.389 0.483    0.759   0.

## Step 4b - Predictive analytics : room price (ADR)


In [16]:
# ------------------------------------------------------------- 4b. PREDICTIVE P2
log("\n== 4b. PREDICTIVE - ADR (positive ADR only; same time split) ==")
r = d[d.adr > 0]
feat2 = [f for f in feat1 if f != "deposit_type"] + ["deposit_type"]
X2 = pd.get_dummies(r[feat2], drop_first=True).astype(float); y2 = r.adr
tr2 = r.arrival_date_year < 2017; te2 = ~tr2
lin = LinearRegression().fit(X2[tr2], y2[tr2]); pl = lin.predict(X2[te2])
rf = RandomForestRegressor(300, min_samples_leaf=5, n_jobs=-1, random_state=SEED).fit(X2[tr2], y2[tr2]); pf = rf.predict(X2[te2])
n, k = te2.sum(), X2.shape[1]
def reg_row(name, p):
    r2 = r2_score(y2[te2], p)
    return dict(Model=name, R2=r2, Adjusted_R2=1 - (1 - r2) * (n - 1) / (n - k - 1),
                RMSE=mean_squared_error(y2[te2], p) ** .5, MAE=mean_absolute_error(y2[te2], p))
reg = pd.DataFrame([dict(Model="Baseline (train mean)", R2=r2_score(y2[te2], np.full(n, y2[tr2].mean())),
                         Adjusted_R2=np.nan, RMSE=mean_squared_error(y2[te2], np.full(n, y2[tr2].mean())) ** .5,
                         MAE=mean_absolute_error(y2[te2], np.full(n, y2[tr2].mean()))),
                    reg_row("Linear regression", pl), reg_row("Random forest regressor", pf)]).round(3)
table(reg, "table_12_adr_model_metrics.csv", index=False); log(reg.to_string(index=False))
# in-sample OLS for coefficient interpretation / diagnostics
ols = sm.OLS(y2, sm.add_constant(X2)).fit()
log("OLS (full clean sample) adj R2 %.3f; Durbin-Watson %.2f" % (ols.rsquared_adj, sm.stats.durbin_watson(ols.resid)))
fig_, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].scatter(y2[te2], pf, s=3, alpha=.25); ax[0].plot([0, 400], [0, 400], "r--")
ax[0].set_xlim(0, 400); ax[0].set_ylim(0, 400); ax[0].set_xlabel("Actual ADR"); ax[0].set_ylabel("Predicted ADR")
ax[0].set_title("Random forest: predicted vs actual (2017)")
ax[1].hist(y2[te2] - pl, bins=60, alpha=.6, label="Linear"); ax[1].hist(y2[te2] - pf, bins=60, alpha=.6, label="Random forest")
ax[1].set_xlabel("Residual (actual - predicted)"); ax[1].legend(); ax[1].set_title("Residuals")
fig("figure_07_adr_model_evaluation.png")


== 4b. PREDICTIVE - ADR (positive ADR only; same time split) ==
                  Model     R2  Adjusted_R2   RMSE    MAE
  Baseline (train mean) -0.120          NaN 57.258 42.139
      Linear regression  0.382        0.381 42.536 30.418
Random forest regressor  0.669        0.669 31.103 22.646
OLS (full clean sample) adj R2 0.443; Durbin-Watson 0.73


## Step 5 Prescriptive analysis


In [17]:
# ------------------------------------------------------------- 5. PRESCRIPTIVE
log("\n== 5. PRESCRIPTIVE - overbooking / deposit scenarios (ASSUMPTIONS, not data) ==")
log("Policy: if predicted cancel probability >= threshold, the hotel accepts one extra booking per flagged booking.")
log("True positive (flagged and cancels): resold room earns resale_rate x booking value (ADR x nights).")
log("False positive (flagged but guest arrives): hotel must walk a guest, costing walk_mult x that booking's ADR (one night).")
te_df = d[te].copy(); te_df["p"] = probs1[best]
te_df["value"] = te_df.adr * te_df.total_nights
thr_grid = np.arange(0.10, 0.91, 0.05)
scen = []
for resale in [0.5, 0.8]:
    for walk in [1, 2, 3]:
        for t in thr_grid:
            f = te_df[te_df.p >= t]
            tp = f[f.is_canceled == 1]; fp = f[f.is_canceled == 0]
            net = resale * tp.value.sum() - (walk * fp.adr.sum())
            scen.append(dict(resale_rate=resale, walk_cost_x_adr=walk, threshold=round(t, 2),
                             flagged=len(f), precision=len(tp) / max(len(f), 1), net_benefit=net))
scen = pd.DataFrame(scen)
table(scen.round(3), "table_13_overbooking_scenarios_full.csv", index=False)
bestrows = scen.loc[scen.groupby(["resale_rate", "walk_cost_x_adr"]).net_benefit.idxmax()].round(2)
table(bestrows, "table_14_best_threshold_per_scenario.csv", index=False)
log(bestrows.to_string(index=False))
plt.figure(figsize=(7, 4))
for (rs, wk), g in scen.groupby(["resale_rate", "walk_cost_x_adr"]):
    if rs == 0.8: plt.plot(g.threshold, g.net_benefit / 1000, marker="o", ms=3, label=f"walk cost {wk}x ADR")
plt.axhline(0, color="k", lw=.8); plt.xlabel("Probability threshold for overbooking"); plt.ylabel("Net benefit (thousand currency units, 2017 test)")
plt.title("Overbooking net benefit by threshold (resale rate 80%)"); plt.legend(fontsize=8)
fig("figure_08_overbooking_threshold.png")

# deposit scenario: assume a deposit rule on high-risk bookings reduces their cancellation by x%
high = te_df[te_df.p >= 0.6]
dep = []
for cut in [0.1, 0.2, 0.3]:
    saved = cut * high[high.is_canceled == 1].value.sum()
    dep.append(dict(assumed_cancel_reduction=cut, high_risk_bookings=len(high),
                    cancelled_among_high_risk=int(high.is_canceled.sum()),
                    revenue_retained=round(saved)))
dep = pd.DataFrame(dep); table(dep, "table_15_deposit_scenarios.csv", index=False)
log("\nDeposit scenario (threshold 0.6; reduction % is an ASSUMPTION to be justified from literature):\n", dep.to_string(index=False))



== 5. PRESCRIPTIVE - overbooking / deposit scenarios (ASSUMPTIONS, not data) ==
Policy: if predicted cancel probability >= threshold, the hotel accepts one extra booking per flagged booking.
True positive (flagged and cancels): resold room earns resale_rate x booking value (ADR x nights).
False positive (flagged but guest arrives): hotel must walk a guest, costing walk_mult x that booking's ADR (one night).
 resale_rate  walk_cost_x_adr  threshold  flagged  precision  net_benefit
         0.5                1       0.25    12094       0.54   1234014.65
         0.5                2       0.30     9375       0.59    678630.42
         0.5                3       0.50     6333       0.64    315986.19
         0.8                1       0.15    20905       0.43   2563451.92
         0.8                2       0.25    12094       0.54   1701523.50
         0.8                3       0.30     9375       0.59   1180858.64

Deposit scenario (threshold 0.6; reduction % is an ASSUMPTION to be j